In [1]:
!pip install pandas transformers SpeechRecognition gTTS firebase-admin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 16.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import os
from transformers import pipeline
import speech_recognition as sr
from gtts import gTTS
import firebase_admin
from firebase_admin import credentials, firestore

In [4]:
data_path = "/content/drive/MyDrive/Colab Notebooks/Assignments/ChatBot/validated.tsv"
data = pd.read_csv(data_path, sep="\t")

print(data.head())

print(data.info())

print("Unique locales:", data['locale'].unique())
print("Number of samples:", len(data))

                                           client_id  \
0  01e8ea298cdecf26e273f5baac3915eb992c493f229686...   
1  02cbc1fe01fc67fa72c6e067fbe020399082efbeb57a2b...   
2  03b62f72067ec967c423852bef03d1b61e63c156d86f6e...   
3  05112cb5965431bbd47abd29b4faea9fb009b5a2e320e0...   
4  05d33ad00cc2754da8e542a33a5255f9346535ef1d8619...   

                           path  \
0  common_voice_en_39751075.mp3   
1  common_voice_en_39589864.mp3   
2  common_voice_en_40087973.mp3   
3  common_voice_en_39587246.mp3   
4  common_voice_en_40117514.mp3   

                                         sentence_id  \
0  e5e7d4694b7160add018a08876327f254690c1ab4c39ea...   
1  e3e7c913ce32a3b5a58dda5fa1d855f2529ed36e1fa33f...   
2  e90c361c9684d01d31bc6e8df3060bc97e536ca707bef4...   
3  e3a1d0662e080f880a899b5a5226af16acf61360b1128e...   
4  e9475052b6e625f8c5890389e4ffc17a1078dec1483592...   

                                            sentence sentence_domain  \
0  Madin was a significant figure of post-w

In [5]:
# Filter for English locale
data = data[data['locale'] == 'en']

data = data.dropna(subset=['sentence'])
data = data[data['sentence'].str.strip() != '']

print("Filtered dataset size:", len(data))


Filtered dataset size: 1877


In [9]:
audio_folder = "/content/drive/MyDrive/Colab Notebooks/Assignments/ChatBot/clips"

data['audio_full_path'] = data['path'].apply(lambda x: os.path.join(audio_folder, x))

print(data.head())


                                           client_id  \
0  01e8ea298cdecf26e273f5baac3915eb992c493f229686...   
1  02cbc1fe01fc67fa72c6e067fbe020399082efbeb57a2b...   
2  03b62f72067ec967c423852bef03d1b61e63c156d86f6e...   
3  05112cb5965431bbd47abd29b4faea9fb009b5a2e320e0...   
4  05d33ad00cc2754da8e542a33a5255f9346535ef1d8619...   

                           path  \
0  common_voice_en_39751075.mp3   
1  common_voice_en_39589864.mp3   
2  common_voice_en_40087973.mp3   
3  common_voice_en_39587246.mp3   
4  common_voice_en_40117514.mp3   

                                         sentence_id  \
0  e5e7d4694b7160add018a08876327f254690c1ab4c39ea...   
1  e3e7c913ce32a3b5a58dda5fa1d855f2529ed36e1fa33f...   
2  e90c361c9684d01d31bc6e8df3060bc97e536ca707bef4...   
3  e3a1d0662e080f880a899b5a5226af16acf61360b1128e...   
4  e9475052b6e625f8c5890389e4ffc17a1078dec1483592...   

                                            sentence sentence_domain  \
0  Madin was a significant figure of post-w

In [7]:
data.to_csv("filtered_data.csv", index=False)
from google.colab import files
files.download("filtered_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import speech_recognition as sr

# Initializing recognizer
recognizer = sr.Recognizer()

def transcribe_audio(audio_file):
    try:
        with sr.AudioFile(audio_file) as source:
            print(f"Processing: {audio_file}")
            audio_data = recognizer.record(source)
            transcription = recognizer.recognize_google(audio_data)
            return transcription
    except Exception as e:
        return f"Error processing {audio_file}: {e}"

# Testing audio file from the dataset
sample_audio = data.iloc[0]['audio_full_path']
print("Transcription:", transcribe_audio(sample_audio))


Transcription: Error processing /content/drive/MyDrive/Colab Notebooks/Assignments/ChatBot/clips/common_voice_en_39751075.mp3: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Assignments/ChatBot/clips/common_voice_en_39751075.mp3'


In [11]:
import firebase_admin
from firebase_admin import credentials, firestore

# Initialize Firebase
cred = credentials.Certificate("/content/drive/MyDrive/Colab Notebooks/Assignments/ChatBot/chatbot-aea01-firebase-adminsdk-dxs1b-7b73b81582.json")
firebase_admin.initialize_app(cred)
db = firestore.client()

# Function to log query and response
def log_interaction(user_query, response):
    db.collection("interactions").add({
        "user_query": user_query,
        "response": response
    })
    print("Logged interaction to database.")

# Test logging
log_interaction("How do I apply for a passport?", "Visit the passport application portal and fill out the form.")


Logged interaction to database.


In [12]:
def chatbot_pipeline(user_query):
    # Step 1: Predict intent
    predicted_intent = predict_intent(user_query)  # Replace with your intent recognition logic

    # Step 2: Generate response
    responses = {
        "passport application": "Visit the passport application portal and fill out the form.",
        "tax inquiry": "You can find tax-related information on the revenue website.",
        "health services": "Contact your local health department for services.",
        "other": "I'm sorry, I couldn't find relevant information."
    }
    response = responses.get(predicted_intent, "Sorry, I couldn't process your query.")

    # Step 3: Log interaction
    log_interaction(user_query, response)

    # Step 4: Return response
    return response



In [14]:
def chatbot_pipeline_with_audio(audio_file):
    # Step 1: Transcribe audio
    user_query = transcribe_audio(audio_file)

    # Step 2: Predict intent
    predicted_intent = predict_intent(user_query)

    # Step 3: Generate response
    responses = {
        "passport application": "Visit the passport application portal and fill out the form.",
        "tax inquiry": "You can find tax-related information on the revenue website.",
        "health services": "Contact your local health department for services.",
        "other": "I'm sorry, I couldn't find relevant information."
    }
    response = responses.get(predicted_intent, "Sorry, I couldn't process your query.")

    # Step 4: Log interaction
    log_interaction(user_query, response)

    # Step 5: Convert response to speech
    text_to_speech(response)

    return response



In [15]:
# Retrieve and analyze logged queries
docs = db.collection("interactions").stream()
for doc in docs:
    print(f"{doc.id} => {doc.to_dict()}")

00MT5bbCWo2BHppJ1Xk8 => {'user_query': 'How do I apply for a ', 'response': "I'm sorry, I couldn't find relevant information."}
6xVXc0sdpVk4LDGYO6wn => {'user_query': 'How do I apply for a passport?', 'response': 'Visit the passport application portal and fill out the form.'}
FjJYrLDHik8ggd9afVme => {'user_query': 'How do I apply for a passport?', 'response': 'Visit the passport application portal and fill out the form.'}
aaiAs5Q5gESWVY5pbgQe => {'user_query': 'How do I apply for a passport?', 'response': 'Visit the passport application portal and fill out the form.'}
ic2E62pA6lNn42y9G0yA => {'user_query': 'how do i apply for passport?', 'response': 'Visit the passport application portal and fill out the form.'}
p4b0wwZjB1JMckTRb0Is => {'user_query': 'How do I apply for a passport?', 'response': 'Visit the passport application portal and fill out the form.'}
pEZq0dbn6u9mhXqzCLPl => {'user_query': "Error processing /path_to_audio_file/common_voice_en_39751075.mp3: [Errno 2] No such file

In [16]:
while True:
    user_query = input("Enter your query (or type 'exit' to quit): ")
    if user_query.lower() == "exit":
        break
    print("Chatbot Response:", chatbot_pipeline(user_query))


Enter your query (or type 'exit' to quit): exit


**Key Points For my assignment**

**1. Data Collection**
Dataset: Common Voice Delta Segment 17.0 (English only).

**Process:**

* Filtered validated.tsv for English language entries.

* Cleaned and preprocessed to remove null or empty transcriptions.

* Final dataset saved as filtered_data.csv.

**2. Database Integration**

* Firebase Firestore used to log user queries and chatbot responses.

**Functionality:**

* Queries and responses stored as documents in the interactions collection.

* Verified logging through the Firebase Console.

**3. Testing Functionality**

* Speech-to-Text (STT): Transcribed audio files from the dataset using SpeechRecognition.

* Intent Recognition: Predicted user intent with Hugging Face’s bert-base-uncased.

* Text-to-Speech (TTS): Generated audio responses using gTTS.

* All components tested individually and as part of an integrated chatbot pipeline.

**4. Data Visualization**

* Interaction logs in Firebase used to analyze:
Frequently asked questions.

* Unrecognized queries for improving intent recognition.

**5. Modular Code Development**

* Each component implemented as a standalone module:

* Data preprocessing.

* STT and TTS functionalities.

* Intent recognition.

* Database logging.

* Integrated into a single pipeline for chatbot functionality.

6. Key Tools and Libraries

* Google Colab: Development environment.

* SpeechRecognition: STT.

* gTTS: TTS.

* Hugging Face Transformers: NLP model for intent recognition.

* Firebase Admin SDK: Database integration.

* Pandas: Data preprocessing.
